In [2]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset/ait_ads_wazuh_raw.parquet")

df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()

(2493428, 15)


,scenario,timestamp,agent_id,agent_name,agent_ip,hostname,program,location,full_log,rule_id,rule_level,rule_description,rule_groups,rule_groups_str,y_priority
0,fox,2022-01-15 02:32:32+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
1,fox,2022-01-15 02:32:32+00:00,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,/var/log/syslog,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
2,fox,2022-01-15 02:32:37+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
3,fox,2022-01-15 02:32:42+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:42 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low
4,fox,2022-01-15 02:32:47+00:00,18,wazuh-client,172.17.131.81,mail,freshclam,/var/log/syslog,Jan 15 02:32:47 mail freshclam[29266]: Sat Jan...,52507,3,ClamAV database update,"[clamd, freshclam, virus]","clamd,freshclam,virus",low


In [3]:
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek

df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
df["is_night"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)

In [4]:
def is_internal(ip):
    if pd.isna(ip):
        return 0
    return int(
        ip.startswith("10.") or
        ip.startswith("192.168.") or
        ip.startswith("172.")
    )

df["is_internal_ip"] = df["agent_ip"].apply(is_internal)

In [5]:
df["has_400"] = df["full_log"].str.contains(" 400 ", na=False).astype(int)
df["has_401"] = df["full_log"].str.contains(" 401 ", na=False).astype(int)
df["has_403"] = df["full_log"].str.contains(" 403 ", na=False).astype(int)
df["has_500"] = df["full_log"].str.contains(" 500 ", na=False).astype(int)

df["has_sql"] = df["full_log"].str.contains("SELECT|UNION|OR 1=1", case=False, na=False).astype(int)
df["has_admin"] = df["full_log"].str.contains("/admin|wp-admin", case=False, na=False).astype(int)
df["has_wp"] = df["full_log"].str.contains("wp-", case=False, na=False).astype(int)

In [6]:
df["grp_attack"] = df["rule_groups_str"].str.contains("attack", na=False).astype(int)
df["grp_recon"] = df["rule_groups_str"].str.contains("recon", na=False).astype(int)
df["grp_scan"] = df["rule_groups_str"].str.contains("scan", na=False).astype(int)
df["grp_auth"] = df["rule_groups_str"].str.contains("auth", na=False).astype(int)

In [7]:
new_cols = [
    "hour", "dayofweek", "is_weekend", "is_night",
    "is_internal_ip",
    "has_400", "has_401", "has_403", "has_500",
    "has_sql", "has_admin", "has_wp",
    "grp_attack", "grp_recon", "grp_scan", "grp_auth"
]

print("Количество новых признаков:", len(new_cols))
print()
print(df[new_cols].head(10))

Количество новых признаков: 16

   hour  dayofweek  is_weekend  is_night  is_internal_ip  has_400  has_401  \
0     2          5           1         1               1        0        0   
1     2          5           1         1               1        0        0   
2     2          5           1         1               1        0        0   
3     2          5           1         1               1        0        0   
4     2          5           1         1               1        0        0   
5     2          5           1         1               1        0        0   
6     2          5           1         1               1        0        0   
7     2          5           1         1               1        0        0   
8     2          5           1         1               1        0        0   
9     3          5           1         1               1        0        0   

   has_403  has_500  has_sql  has_admin  has_wp  grp_attack  grp_recon  \
0        0        0        0       

In [8]:
for col in new_cols:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).head(10))


===== hour =====
hour
7     876490
11    492258
12    482266
10     72795
9      65208
13     61688
14     60153
15     56716
8      52301
16     46110
Name: count, dtype: int64

===== dayofweek =====
dayofweek
1    889158
6    582388
0    528204
5    176336
4    168529
3     83955
2     64858
Name: count, dtype: int64

===== is_weekend =====
is_weekend
0    1734704
1     758724
Name: count, dtype: int64

===== is_night =====
is_night
0    2422342
1      71086
Name: count, dtype: int64

===== is_internal_ip =====
is_internal_ip
1    2493428
Name: count, dtype: int64

===== has_400 =====
has_400
0    2493428
Name: count, dtype: int64

===== has_401 =====
has_401
0    2493426
1          2
Name: count, dtype: int64

===== has_403 =====
has_403
0    2490758
1       2670
Name: count, dtype: int64

===== has_500 =====
has_500
0    2493361
1         67
Name: count, dtype: int64

===== has_sql =====
has_sql
0    2491231
1       2197
Name: count, dtype: int64

===== has_admin =====
has_admin
0

In [9]:
from pathlib import Path

FEATURE_PATH = Path("/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset/wazuh_features.parquet")

df.to_parquet(FEATURE_PATH, index=False)

print("Сохранено:")
print(FEATURE_PATH)
print("Размер:", df.shape)

Сохранено:
/home/chupchik/Рабочий стол/fisrt_stepD/The_final_recomendation/output_dataset/wazuh_features.parquet
Размер: (2493428, 31)
